In [1]:
# 세팅
import pandas as pd
pd.plotting.register_matplotlib_converters()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [2]:
trial_usage_summary = pd.read_csv('/kaggle/input/datasets/mam4182/nextwave-dataset/trial_usage_summary.csv')
segment_retention = pd.read_csv('/kaggle/input/datasets/mam4182/nextwave-dataset/segment_retention.csv')
landing_event_log = pd.read_csv('/kaggle/input/datasets/mam4182/nextwave-dataset/landing_event_log.csv')
funnel_summary = pd.read_csv('/kaggle/input/datasets/mam4182/nextwave-dataset/funnel_summary.csv')
customer_feedback_summary = pd.read_csv('/kaggle/input/datasets/mam4182/nextwave-dataset/customer_feedback_summary.csv')
channel_performance = pd.read_csv('/kaggle/input/datasets/mam4182/nextwave-dataset/channel_performance.csv')

In [3]:
from IPython.display import display

# 데이터프레임 리스트와 이름 설정
dfs = [
    ("Trial_Usage", trial_usage_summary),
    ("Segment_Retention", segment_retention),
    ("Landing_Event_Log", landing_event_log),
    ("Funnel_Summary", funnel_summary),
    ("Customer_Feedback_Summary", customer_feedback_summary),
    ("Channel_Performance", channel_performance)
]

for title, df in dfs:
    print(f" [ {title} ]")
    display(df)
    print("\n")

 [ Trial_Usage ]


,기능,기능 사용자 수,평균 사용 횟수,유료 전환율
0,일정관리,2840,8.2,24.0%
1,협업메모,1920,4.1,17.5%
2,팀 초대,740,2.7,36.0%
3,알림 자동화,510,1.9,41.0%




 [ Segment_Retention ]


,고객 유형,4주차 유지율
0,대학생,21%
1,직장인 개인사용자,38%
2,프리랜서,34%
3,소규모 팀 사용자,49%




 [ Landing_Event_Log ]


...

,month,방문 유저 수,회원가입 유저,무료 체험 유저,유료 결제 유저,한달 유지
0,1월,52000,4420,2431,972,1945
1,2월,54000,4212,2274,880,1801
2,3월,55500,3885,1942,731,1492
3,4월,56000,3528,1623,584,1176




 [ Customer_Feedback_Summary ]


,이슈 유형,비중
0,가입 절차 번거로움,28%
1,무료체험 가치 체감 부족,24%
2,요금제 이해 어려움,18%
3,팀 기능 이해 부족,16%
4,알림 과다,14%




 [ Channel_Performance ]


,month,유입 채널,방문자 수,회원가입 전환율,무료체험 시작률,유료 전환율
0,4월,검색,18000,8.1%,5.0%,1.8%
1,4월,SNS,17500,4.3%,2.1%,0.7%
2,4월,자연 유입,11000,9.5%,6.4%,2.2%
3,4월,지인 추천,9500,10.2%,7.1%,2.8%


## 개념 정리: `이탈률`

### **이탈률 = (직전단계 유저수 - 현재단계 유저수)/직전단계 유저수**

### --> Landing_Event_Log의 첫 단계인 랜딩페이지 진입 단계의 이탈률은 계산 할 수 없음.

In [4]:
# 데이터 파악:
# 제공된 데이터는 크기가 작아 눈으로 직접 확인 가능
# 결측치나 문자 데이터를 골라내는 코드 작성할 필요 없음(오버 엔지니어링)

# 확인 결과:
# Landing_Event_Log의 랜딩페이지 진입에 '-' 비정형 결측치 값이 존재하는데,
# 이는 위에서 개념 설명한 것처럼 구할 수 없는 값임.
# 또한 % 등의 값들은 연산을 위해서 소수점으로 변환할 필요 있음

In [5]:
for name, df in dfs:
    for col in df.columns:
        if df[col].astype(str).str.contains('%').any():
            # % 제거
            temp_col = df[col].astype(str).str.replace('%', '', regex=False)
            # 숫자로 변환 (변환 안 되는 '-'은 NaN 처리)
            # 100으로 나누어 소수점 변환
            df[col] = pd.to_numeric(temp_col, errors='coerce') / 100
            
            print(f"[{name}]의 '{col}' 컬럼 변환 완료\n")

[Trial_Usage]의 '유료 전환율' 컬럼 변환 완료

[Segment_Retention]의 '4주차 유지율' 컬럼 변환 완료

[Landing_Event_Log]의 '이탈률' 컬럼 변환 완료

[Customer_Feedback_Summary]의 '비중' 컬럼 변환 완료

[Channel_Performance]의 '회원가입 전환율' 컬럼 변환 완료

[Channel_Performance]의 '무료체험 시작률' 컬럼 변환 완료

[Channel_Performance]의 '유료 전환율' 컬럼 변환 완료



In [6]:
for title, df in dfs:
    print(f" [ {title} ]")
    display(df)
    print("\n")

 [ Trial_Usage ]


,기능,기능 사용자 수,평균 사용 횟수,유료 전환율
0,일정관리,2840,8.2,0.240
1,협업메모,1920,4.1,0.175
2,팀 초대,740,2.7,0.360
3,알림 자동화,510,1.9,0.410




 [ Segment_Retention ]


,고객 유형,4주차 유지율
0,대학생,0.21
1,직장인 개인사용자,0.38
2,프리랜서,0.34
3,소규모 팀 사용자,0.49




 [ Landing_Event_Log ]


...

,month,방문 유저 수,회원가입 유저,무료 체험 유저,유료 결제 유저,한달 유지
0,1월,52000,4420,2431,972,1945
1,2월,54000,4212,2274,880,1801
2,3월,55500,3885,1942,731,1492
3,4월,56000,3528,1623,584,1176




 [ Customer_Feedback_Summary ]


,이슈 유형,비중
0,가입 절차 번거로움,0.28
1,무료체험 가치 체감 부족,0.24
2,요금제 이해 어려움,0.18
3,팀 기능 이해 부족,0.16
4,알림 과다,0.14




 [ Channel_Performance ]


,month,유입 채널,방문자 수,회원가입 전환율,무료체험 시작률,유료 전환율
0,4월,검색,18000,0.081,0.050,0.018
1,4월,SNS,17500,0.043,0.021,0.007
2,4월,자연 유입,11000,0.095,0.064,0.022
3,4월,지인 추천,9500,0.102,0.071,0.028


In [7]:
# %형식 데이터 가공 가능한 수치형 데이터로 변환 완료
# NaN값은 미팅에서 어떤 방식으로 처리 할 건지 논의 예정

# 교차 검증 필요